# 09 - Pontuação Necessária para Cada Objetivo

Quantos pontos precisa para: ser campeão, ir à Libertadores (top 4), escapar do rebaixamento?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df[df["ano"] >= 2003].copy()

In [2]:
def classificacao_final(df_season):
    teams = set(df_season["mandante"].unique()) | set(df_season["visitante"].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    
    for _, jogo in df_season.iterrows():
        m, v = jogo["mandante"], jogo["visitante"]
        gm, gv = jogo["mandante_Placar"], jogo["visitante_Placar"]
        if pd.isna(gm) or pd.isna(gv):
            continue
        gm, gv = int(gm), int(gv)
        saldo[m] += gm - gv
        saldo[v] += gv - gm
        if gm > gv:
            pontos[m] += 3; vitorias[m] += 1
        elif gm < gv:
            pontos[v] += 3; vitorias[v] += 1
        else:
            pontos[m] += 1; pontos[v] += 1
    
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    return [(t, pontos[t], i+1) for i, t in enumerate(ranking)]

# Coletar dados de cada temporada
dados = []
for ano in sorted(df["ano"].unique()):
    df_s = df[df["ano"] == ano]
    df_s_clean = df_s.dropna(subset=["rodata"])
    if df_s_clean["rodata"].astype(int).max() < 30:
        continue
    classif = classificacao_final(df_s)
    n = len(classif)
    
    pts_campeao = classif[0][1]
    pts_4 = classif[3][1] if n > 3 else None
    pts_16 = classif[min(15, n-1)][1]
    pts_17 = classif[min(16, n-1)][1]
    
    dados.append({
        "ano": ano,
        "Campeão": pts_campeao,
        "4º lugar (Libertadores)": pts_4,
        "16º lugar (escapa Z4)": pts_16,
        "17º lugar (entra Z4)": pts_17
    })

df_pts = pd.DataFrame(dados)
df_pts

,ano,Campeão,4º lugar (Libertadores),16º lugar (escapa Z4),17º lugar (entra Z4)
0,2003,100,73,57,56
1,2004,89,79,54,54
2,2005,81,70,53,52
3,2006,78,64,44,39
4,2007,77,61,45,44
5,2008,75,65,44,44
6,2009,67,62,46,45
7,2010,71,63,42,42
8,2011,71,61,43,41
9,2012,77,66,45,41


In [3]:
# Gráfico principal: faixas de pontuação ao longo dos anos
fig = go.Figure()

cores = {"Campeão": "#f1c40f", "4º lugar (Libertadores)": "#27ae60",
         "16º lugar (escapa Z4)": "#3498db", "17º lugar (entra Z4)": "#e74c3c"}

for col, cor in cores.items():
    fig.add_trace(go.Scatter(
        x=df_pts["ano"], y=df_pts[col],
        mode="lines+markers", name=col,
        line=dict(color=cor, width=3),
        marker=dict(size=7),
        hovertemplate=f"{col}<br>%{{x}}: %{{y}} pts<extra></extra>"
    ))

fig.update_layout(
    title="Pontuação por Objetivo ao Longo dos Anos<br><sup>Brasileirão Série A (2003-presente)</sup>",
    xaxis_title="Ano",
    yaxis_title="Pontos",
    template="plotly_white",
    width=950,
    height=550,
    legend=dict(x=0.01, y=0.99)
)

fig.show()
fig.write_html("../charts/pontuacao_objetivos.html", include_plotlyjs="cdn")
print("Gráfico exportado: charts/pontuacao_objetivos.html")

Gráfico exportado: charts/pontuacao_objetivos.html


In [4]:
# Estatísticas resumidas
print("=== Pontuação Histórica (média ± desvio) ===")
for col in ["Campeão", "4º lugar (Libertadores)", "16º lugar (escapa Z4)", "17º lugar (entra Z4)"]:
    media = df_pts[col].mean()
    std = df_pts[col].std()
    minimo = df_pts[col].min()
    maximo = df_pts[col].max()
    print(f"\n{col}:")
    print(f"  Média: {media:.0f} ± {std:.0f}")
    print(f"  Min: {int(minimo)} | Max: {int(maximo)}")

=== Pontuação Histórica (média ± desvio) ===

Campeão:
  Média: 80 ± 9
  Min: 67 | Max: 103

4º lugar (Libertadores):
  Média: 66 ± 5
  Min: 61 | Max: 79

16º lugar (escapa Z4):
  Média: 45 ± 5
  Min: 39 | Max: 57

17º lugar (entra Z4):
  Média: 44 ± 5
  Min: 36 | Max: 56


In [5]:
# Box plot das distribuições
df_melt = df_pts.melt(id_vars="ano", value_vars=list(cores.keys()),
                       var_name="Objetivo", value_name="Pontos")

fig2 = px.box(df_melt, x="Objetivo", y="Pontos",
              color="Objetivo",
              color_discrete_map=cores,
              title="Distribuição de Pontos por Objetivo<br><sup>Brasileirão (2003-presente)</sup>",
              points="all")
fig2.update_layout(template="plotly_white", width=800, height=500, showlegend=False)

fig2.show()
fig2.write_html("../charts/pontuacao_box.html", include_plotlyjs="cdn")
print("Gráfico exportado: charts/pontuacao_box.html")

Gráfico exportado: charts/pontuacao_box.html
